In [2]:
import pandas as pd
import pm4py

df = pd.read_csv("../data/BPI Challenge 2017.csv")
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], format="mixed")
df = pm4py.format_dataframe(df, case_id="case:concept:name",
                            activity_key="concept:name", timestamp_key="time:timestamp")
log = df[(df["EventOrigin"] != "Workflow") | (df["lifecycle:transition"] == "start")]
log = log.sort_values(["case:concept:name", "time:timestamp"], kind="stable")
prev = log.groupby("case:concept:name")["concept:name"].shift()
log = log[(log["concept:name"] != prev).fillna(True).astype(bool)]
print("Events:", len(log))

Events: 552624


In [3]:
DECISION = ["O_Accepted", "A_Cancelled", "A_Denied"]
first = log[log["concept:name"].isin(DECISION)].groupby("case:concept:name").first()
y = first["concept:name"]
t_decision = first["time:timestamp"]

a = log.join(t_decision.rename("t_dec"), on="case:concept:name")
before = a[a["time:timestamp"] < a["t_dec"]]
offers = before[before["concept:name"] == "O_Create Offer"]
last_offer = offers.groupby("case:concept:name").last()
case_attr = log.groupby("case:concept:name")[["case:ApplicationType", "case:RequestedAmount"]].first()
start = log.groupby("case:concept:name")["time:timestamp"].min()

X = pd.DataFrame(index=y.index)
X["RequestedAmount"] = case_attr["case:RequestedAmount"]
X["LimitRaise"] = (case_attr["case:ApplicationType"] == "Limit raise").astype(int)
for c in ["OfferedAmount", "MonthlyCost", "NumberOfTerms"]:
    X[c] = last_offer[c]
X["NumOffers"] = offers.groupby("case:concept:name").size()
for act, col in [("O_Returned", "OfferReturned"), ("A_Incomplete", "Incomplete")]:
    occurred = before[before["concept:name"] == act]["case:concept:name"].unique()
    X[col] = X.index.isin(occurred).astype(int)
X["DaysToDecision"] = (t_decision - start).dt.total_seconds() / 86400
# workflow features: how often was a work item started before the decision?
for act, col in [("W_Call after offers", "CallsAfterOffers"),
                 ("W_Call incomplete files", "CallsIncomplete"),
                 ("W_Validate application", "Validations"),
                 ("W_Assess potential fraud", "FraudChecks")]:
    X[col] = before[before["concept:name"] == act].groupby("case:concept:name").size()
X = X.fillna(0)
print(y.value_counts())

concept:name
O_Accepted     17228
A_Cancelled    10431
A_Denied        3752
Name: count, dtype: Int64


In [4]:
cs = last_offer["CreditScore"].reindex(y.index).fillna(0)
(cs > 0).groupby(y).mean().round(3)

concept:name
A_Cancelled    0.000
A_Denied       0.000
O_Accepted     0.819
Name: CreditScore, dtype: float64

In [5]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
clf = DecisionTreeClassifier(max_depth=2, min_samples_leaf=200, random_state=42).fit(X_train, y_train)
pred = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3),
      "| Baseline:", round(y.value_counts(normalize=True).max(), 3))
print(classification_report(y_test, pred, digits=2, zero_division=0))
print(export_text(clf, feature_names=list(X.columns)))
print(pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False).round(3).head(5))

Accuracy: 0.849 | Baseline: 0.548
              precision    recall  f1-score   support

 A_Cancelled       0.98      0.90      0.94      3129
    A_Denied       0.00      0.00      0.00      1126
  O_Accepted       0.79      1.00      0.88      5169

    accuracy                           0.85      9424
   macro avg       0.59      0.63      0.61      9424
weighted avg       0.76      0.85      0.80      9424

|--- OfferReturned <= 0.50
|   |--- DaysToDecision <= 28.91
|   |   |--- class: A_Cancelled
|   |--- DaysToDecision >  28.91
|   |   |--- class: A_Cancelled
|--- OfferReturned >  0.50
|   |--- Incomplete <= 0.50
|   |   |--- class: O_Accepted
|   |--- Incomplete >  0.50
|   |   |--- class: O_Accepted

OfferReturned      0.957
Incomplete         0.039
DaysToDecision     0.004
LimitRaise         0.000
RequestedAmount    0.000
dtype: float64


In [6]:
leaf = pd.Series(clf.apply(X_train), index=X_train.index)
pd.crosstab(leaf, y_train, normalize="index").round(3)

concept:name,A_Cancelled,A_Denied,O_Accepted
row_0,,,
2,0.871,0.129,0.000
3,0.999,0.001,0.000
5,0.009,0.322,0.669
6,0.059,0.090,0.851
